# 00 Eda Baseline

Baseline EDA for the MLC churn dataset.

This script focuses on reproducible tabular outputs:
- dataset structure and basic statistics
- missing-value analysis
- churn class balance
- numeric correlations and redundant-feature candidates
- a simple, model-ready encoded dataset for the next phases

Run from the project root or from this folder:
    python Phase_0_EDA/00_eda_baseline.py

Notebook generated from `00_eda_baseline.py`. Run all cells to reproduce the Phase 0 outputs.

## Setup and helper functions

Run this cell first to import libraries, locate the dataset, and define reusable EDA helpers.

In [1]:
from __future__ import annotations

import io
from pathlib import Path

import numpy as np
import pandas as pd


RANDOM_SEED = 1234
TARGET = "churn"
POSITIVE_CLASS = "yes"
HIGH_CORRELATION_THRESHOLD = 0.95


def project_root() -> Path:
    """Return project root when running from either the repo root or Phase_0_EDA."""
    current = Path.cwd().resolve()
    for candidate in [current, current.parent]:
        if (candidate / "mlc_churn.csv").exists() and (candidate / "Phase_0_EDA").exists():
            return candidate
    if "__file__" in globals():
        return Path(__file__).resolve().parents[1]
    raise FileNotFoundError("Cannot locate project root. Run the notebook from the repo root or Phase_0_EDA.")


ROOT_DIR = project_root()
DATA_PATH = ROOT_DIR / "mlc_churn.csv"
PHASE_DIR = ROOT_DIR / "Phase_0_EDA"
OUTPUT_DIR = PHASE_DIR / "outputs"


def section(title: str) -> None:
    line = "=" * 80
    print(f"\n{line}\n{title}\n{line}")


def load_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    return pd.read_csv(path)


def split_feature_types(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    feature_df = df.drop(columns=[TARGET], errors="ignore")
    numeric_features = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    return numeric_features, categorical_features


def make_missing_report(df: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_percent": df.isna().mean() * 100,
            "dtype": df.dtypes.astype(str),
            "n_unique": df.nunique(dropna=True),
        }
    )
    return report.sort_values(["missing_count", "missing_percent"], ascending=False)


def make_churn_distribution(df: pd.DataFrame) -> pd.DataFrame:
    counts = df[TARGET].value_counts(dropna=False)
    distribution = pd.DataFrame(
        {
            "count": counts,
            "percent": counts / len(df) * 100,
        }
    )
    distribution.index.name = TARGET
    return distribution


def make_correlation_pairs(correlation_matrix: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    columns = correlation_matrix.columns.tolist()
    for idx, left in enumerate(columns):
        for right in columns[idx + 1 :]:
            value = correlation_matrix.loc[left, right]
            rows.append(
                {
                    "feature_1": left,
                    "feature_2": right,
                    "correlation": value,
                    "abs_correlation": abs(value),
                }
            )
    return pd.DataFrame(rows).sort_values("abs_correlation", ascending=False)


def encode_target(df: pd.DataFrame) -> pd.Series:
    return df[TARGET].astype(str).str.lower().eq(POSITIVE_CLASS).astype(int)


def make_numeric_target_correlation(
    df: pd.DataFrame, numeric_features: list[str]
) -> pd.DataFrame:
    target_binary = encode_target(df)
    rows = []
    for column in numeric_features:
        corr = df[column].corr(target_binary)
        rows.append(
            {
                "feature": column,
                "correlation_with_churn_yes": corr,
                "abs_correlation": abs(corr),
            }
        )
    return pd.DataFrame(rows).sort_values("abs_correlation", ascending=False)


def make_categorical_churn_rates(
    df: pd.DataFrame, categorical_features: list[str]
) -> pd.DataFrame:
    rows = []
    for column in categorical_features:
        grouped = (
            df.groupby(column, dropna=False)[TARGET]
            .agg(
                total="size",
                churn_yes=lambda values: (values.astype(str).str.lower() == POSITIVE_CLASS).sum(),
            )
            .reset_index()
        )
        grouped["churn_rate"] = grouped["churn_yes"] / grouped["total"]
        grouped["feature"] = column
        grouped = grouped.rename(columns={column: "category"})
        rows.append(grouped[["feature", "category", "total", "churn_yes", "churn_rate"]])

    if not rows:
        return pd.DataFrame(columns=["feature", "category", "total", "churn_yes", "churn_rate"])

    return pd.concat(rows, ignore_index=True).sort_values(
        ["feature", "churn_rate", "total"], ascending=[True, False, False]
    )


def make_group_summary(df: pd.DataFrame, numeric_features: list[str]) -> pd.DataFrame:
    if not numeric_features:
        return pd.DataFrame()
    return df.groupby(TARGET)[numeric_features].agg(["mean", "median", "std"]).round(4)


def infer_redundant_charge_features(high_corr_pairs: pd.DataFrame) -> list[str]:
    """Prefer minutes over charge when both are almost perfectly correlated."""
    redundant = set()
    for _, row in high_corr_pairs.iterrows():
        left = str(row["feature_1"])
        right = str(row["feature_2"])
        if row["abs_correlation"] < HIGH_CORRELATION_THRESHOLD:
            continue
        if left.endswith("_charge") and right.endswith("_minutes"):
            redundant.add(left)
        elif right.endswith("_charge") and left.endswith("_minutes"):
            redundant.add(right)
    return sorted(redundant)


def make_model_ready_dataset(
    df: pd.DataFrame,
    categorical_features: list[str],
    redundant_features: list[str],
) -> pd.DataFrame:
    """Create a simple encoded dataset while preserving reproducibility.

    The improved script creates a feature-selected version. Baseline keeps all
    non-target predictors so later phases can decide which variables to remove.
    """
    clean_df = df.copy()

    for column in categorical_features + [TARGET]:
        clean_df[column] = clean_df[column].astype(str).str.strip().str.lower()

    clean_df[f"{TARGET}_binary"] = clean_df[TARGET].eq(POSITIVE_CLASS).astype(int)
    model_df = clean_df.drop(columns=[TARGET])
    model_df = pd.get_dummies(model_df, columns=categorical_features, drop_first=False, dtype=int)
    ordered_columns = [column for column in model_df.columns if column != f"{TARGET}_binary"]
    model_df = model_df[ordered_columns + [f"{TARGET}_binary"]]

    # Keep the redundant columns in the baseline file, but store an explicit flag
    # in a companion text report so Phase 1 can choose a feature-selection policy.
    model_df.attrs["redundant_features"] = redundant_features
    return model_df


def write_feature_summary(
    output_path: Path,
    df: pd.DataFrame,
    numeric_features: list[str],
    categorical_features: list[str],
    churn_distribution: pd.DataFrame,
    missing_report: pd.DataFrame,
    high_corr_pairs: pd.DataFrame,
    redundant_features: list[str],
) -> None:
    lines = [
        "MLC Churn - Phase 0 Baseline EDA Summary",
        "=" * 48,
        "",
        f"Dataset path: {DATA_PATH}",
        f"Rows: {df.shape[0]}",
        f"Columns: {df.shape[1]}",
        f"Target column: {TARGET}",
        f"Random seed for later modeling: {RANDOM_SEED}",
        "",
        "Feature types",
        "-" * 20,
        f"Numeric features ({len(numeric_features)}): {', '.join(numeric_features)}",
        f"Categorical features ({len(categorical_features)}): {', '.join(categorical_features)}",
        "",
        "Class balance",
        "-" * 20,
        churn_distribution.round(4).to_string(),
        "",
        "Missing values",
        "-" * 20,
        f"Total missing cells: {int(missing_report['missing_count'].sum())}",
        "Columns with missing values:",
    ]

    missing_columns = missing_report[missing_report["missing_count"] > 0]
    if missing_columns.empty:
        lines.append("None. No imputation is needed for the current file.")
    else:
        lines.append(missing_columns.to_string())

    lines.extend(
        [
            "",
            "High-correlation feature pairs",
            "-" * 20,
        ]
    )

    top_pairs = high_corr_pairs.head(12)
    lines.append(top_pairs.round(6).to_string(index=False))

    lines.extend(
        [
            "",
            "Redundant-feature recommendation",
            "-" * 20,
        ]
    )
    if redundant_features:
        lines.append(
            "Candidate drop list because charge columns are deterministic or near-deterministic "
            f"from minutes: {', '.join(redundant_features)}"
        )
    else:
        lines.append("No feature pair exceeded the configured high-correlation threshold.")

    lines.extend(
        [
            "",
            "Preprocessing plan for Phase 1",
            "-" * 20,
            "1. Keep the raw CSV as the source of truth.",
            "2. Normalize categorical strings with strip/lowercase.",
            "3. Encode churn as 1 for yes and 0 for no.",
            "4. One-hot encode state, area_code, international_plan, and voice_mail_plan.",
            "5. Use stratified train/test split or stratified cross-validation because churn=yes is the minority class.",
            "6. Consider dropping charge columns after correlation analysis, keeping the corresponding minutes columns.",
            "7. Use random_state=1234 for every split and model initialization.",
        ]
    )

    output_path.write_text("\n".join(lines), encoding="utf-8")


## Execute the EDA workflow

This cell defines the main workflow. The following cell runs it and writes CSV/TXT/PNG outputs.

In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = load_data(DATA_PATH)
numeric_features, categorical_features = split_feature_types(df)

In [13]:
section("1. Dataset overview")
print(f"Data path: {DATA_PATH}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nColumns:")
print(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}).to_string(index=False))
print("\nHead:")
print(df.head().to_string(index=False))
print("\nTail:")
print(df.tail().to_string(index=False))

info_buffer = io.StringIO()
df.info(buf=info_buffer)
(OUTPUT_DIR / "data_info.txt").write_text(info_buffer.getvalue(), encoding="utf-8")


1. Dataset overview
Data path: D:\Folder F\phamtuananh@23020010\UET.iSEML\2026.DAE.MLC Churn\mlc_churn.csv
Shape: 5000 rows x 20 columns

Columns:
                       column   dtype
                        state  object
               account_length   int64
                    area_code  object
           international_plan  object
              voice_mail_plan  object
        number_vmail_messages   int64
            total_day_minutes float64
              total_day_calls   int64
             total_day_charge float64
            total_eve_minutes float64
              total_eve_calls   int64
             total_eve_charge float64
          total_night_minutes float64
            total_night_calls   int64
           total_night_charge float64
           total_intl_minutes float64
             total_intl_calls   int64
            total_intl_charge float64
number_customer_service_calls   int64
                        churn  object

Head:
state  account_length     area_code internation

1491

In [14]:
section("2. Summary statistics")
numeric_summary = df[numeric_features].describe().T
categorical_summary = df[categorical_features + [TARGET]].describe().T
print("\nNumeric summary:")
print(numeric_summary.round(4).to_string())
print("\nCategorical summary:")
print(categorical_summary.to_string())
numeric_summary.to_csv(OUTPUT_DIR / "numeric_summary.csv")
categorical_summary.to_csv(OUTPUT_DIR / "categorical_summary.csv")


2. Summary statistics

Numeric summary:
                                count      mean      std  min      25%     50%     75%     max
account_length                 5000.0  100.2586  39.6946  1.0   73.000  100.00  127.00  243.00
number_vmail_messages          5000.0    7.7552  13.5464  0.0    0.000    0.00   17.00   52.00
total_day_minutes              5000.0  180.2889  53.8947  0.0  143.700  180.10  216.20  351.50
total_day_calls                5000.0  100.0294  19.8312  0.0   87.000  100.00  113.00  165.00
total_day_charge               5000.0   30.6497   9.1621  0.0   24.430   30.62   36.75   59.76
total_eve_minutes              5000.0  200.6366  50.5513  0.0  166.375  201.00  234.10  363.70
total_eve_calls                5000.0  100.1910  19.8265  0.0   87.000  100.00  114.00  170.00
total_eve_charge               5000.0   17.0543   4.2968  0.0   14.140   17.09   19.90   30.91
total_night_minutes            5000.0  200.3916  50.5278  0.0  166.900  200.40  234.70  395.00
total_nig

In [5]:
section("3. Missing values")
missing_report = make_missing_report(df)
print(missing_report.to_string())
missing_report.to_csv(OUTPUT_DIR / "missing_values.csv")


3. Missing values
                               missing_count  missing_percent    dtype  n_unique
state                                      0              0.0   object        51
account_length                             0              0.0    int64       218
area_code                                  0              0.0   object         3
international_plan                         0              0.0   object         2
voice_mail_plan                            0              0.0   object         2
number_vmail_messages                      0              0.0    int64        48
total_day_minutes                          0              0.0  float64      1961
total_day_calls                            0              0.0    int64       123
total_day_charge                           0              0.0  float64      1961
total_eve_minutes                          0              0.0  float64      1879
total_eve_calls                            0              0.0    int64       126
total_eve

In [6]:
section("4. Churn class distribution")
churn_distribution = make_churn_distribution(df)
print(churn_distribution.round(4).to_string())
churn_distribution.to_csv(OUTPUT_DIR / "churn_distribution.csv")

churn_yes_rate = churn_distribution.loc[POSITIVE_CLASS, "percent"] if POSITIVE_CLASS in churn_distribution.index else 0
if churn_yes_rate < 20:
    print(f"\nClass imbalance detected: churn=yes is only {churn_yes_rate:.2f}% of the data.")
else:
    print(f"\nClass balance is moderate: churn=yes is {churn_yes_rate:.2f}% of the data.")


4. Churn class distribution
       count  percent
churn                
no      4293    85.86
yes      707    14.14

Class imbalance detected: churn=yes is only 14.14% of the data.


In [7]:
section("5. Feature type and cardinality")
feature_type_summary = pd.DataFrame(
    {
        "feature": numeric_features + categorical_features,
        "type": ["numeric"] * len(numeric_features) + ["categorical"] * len(categorical_features),
        "n_unique": [df[column].nunique(dropna=True) for column in numeric_features + categorical_features],
    }
)
print(feature_type_summary.to_string(index=False))
feature_type_summary.to_csv(OUTPUT_DIR / "feature_type_summary.csv", index=False)


5. Feature type and cardinality
                      feature        type  n_unique
               account_length     numeric       218
        number_vmail_messages     numeric        48
            total_day_minutes     numeric      1961
              total_day_calls     numeric       123
             total_day_charge     numeric      1961
            total_eve_minutes     numeric      1879
              total_eve_calls     numeric       126
             total_eve_charge     numeric      1659
          total_night_minutes     numeric      1853
            total_night_calls     numeric       131
           total_night_charge     numeric      1028
           total_intl_minutes     numeric       170
             total_intl_calls     numeric        21
            total_intl_charge     numeric       170
number_customer_service_calls     numeric        10
                        state categorical        51
                    area_code categorical         3
           international_plan c

In [8]:
section("6. Correlation analysis")
correlation_matrix = df[numeric_features].corr()
correlation_matrix.to_csv(OUTPUT_DIR / "correlation_matrix.csv")
correlation_pairs = make_correlation_pairs(correlation_matrix)
correlation_pairs.to_csv(OUTPUT_DIR / "correlation_pairs.csv", index=False)

high_corr_pairs = correlation_pairs[
    correlation_pairs["abs_correlation"] >= HIGH_CORRELATION_THRESHOLD
].copy()
high_corr_pairs.to_csv(OUTPUT_DIR / "high_correlation_pairs.csv", index=False)
print("Top absolute correlations:")
print(correlation_pairs.head(12).round(6).to_string(index=False))
print(f"\nPairs with abs(correlation) >= {HIGH_CORRELATION_THRESHOLD}:")
if high_corr_pairs.empty:
    print("None")
else:
    print(high_corr_pairs.round(6).to_string(index=False))


6. Correlation analysis
Top absolute correlations:
            feature_1          feature_2  correlation  abs_correlation
    total_day_minutes   total_day_charge     1.000000         1.000000
    total_eve_minutes   total_eve_charge     1.000000         1.000000
  total_night_minutes total_night_charge     0.999999         0.999999
   total_intl_minutes  total_intl_charge     0.999993         0.999993
       account_length    total_day_calls     0.028240         0.028240
  total_night_minutes  total_night_calls     0.026972         0.026972
    total_night_calls total_night_charge     0.026949         0.026949
number_vmail_messages   total_eve_charge     0.019496         0.019496
number_vmail_messages  total_eve_minutes     0.019490         0.019490
     total_day_charge total_intl_minutes    -0.019490         0.019490
    total_day_minutes total_intl_minutes    -0.019486         0.019486
     total_day_charge  total_intl_charge    -0.019419         0.019419

Pairs with abs(correlati

In [9]:
section("7. Target association")
numeric_target_corr = make_numeric_target_correlation(df, numeric_features)
categorical_churn_rates = make_categorical_churn_rates(df, categorical_features)
group_summary = make_group_summary(df, numeric_features)

print("\nNumeric correlation with churn=yes:")
print(numeric_target_corr.round(6).to_string(index=False))
numeric_target_corr.to_csv(OUTPUT_DIR / "numeric_target_correlation.csv", index=False)
categorical_churn_rates.to_csv(OUTPUT_DIR / "categorical_churn_rates.csv", index=False)
group_summary.to_csv(OUTPUT_DIR / "numeric_summary_by_churn.csv")

print("\nHighest categorical churn-rate groups with at least 20 records:")
filtered_rates = categorical_churn_rates[categorical_churn_rates["total"] >= 20]
print(filtered_rates.head(15).round(4).to_string(index=False))



7. Target association

Numeric correlation with churn=yes:
                      feature  correlation_with_churn_yes  abs_correlation
number_customer_service_calls                    0.212564         0.212564
            total_day_minutes                    0.207705         0.207705
             total_day_charge                    0.207700         0.207700
        number_vmail_messages                   -0.097633         0.097633
            total_eve_minutes                    0.089288         0.089288
             total_eve_charge                    0.089282         0.089282
           total_intl_minutes                    0.063285         0.063285
            total_intl_charge                    0.063275         0.063275
             total_intl_calls                   -0.046893         0.046893
          total_night_minutes                    0.045677         0.045677
           total_night_charge                    0.045673         0.045673
               account_length           

In [15]:
section("8. Baseline preprocessing output")
redundant_features = infer_redundant_charge_features(high_corr_pairs)
model_ready_df = make_model_ready_dataset(df, categorical_features, redundant_features)
model_ready_path = OUTPUT_DIR / "preprocessed_baseline.csv"
model_ready_df.to_csv(model_ready_path, index=False)

write_feature_summary(
    OUTPUT_DIR / "feature_summary.txt",
    df,
    numeric_features,
    categorical_features,
    churn_distribution,
    missing_report,
    high_corr_pairs,
    redundant_features,
)

print(f"Saved model-ready baseline data: {model_ready_path}")
print(f"Preprocessed shape: {model_ready_df.shape[0]} rows x {model_ready_df.shape[1]} columns")
if redundant_features:
    print(f"Recommended redundant features to review/drop later: {', '.join(redundant_features)}")

section("Done")
print(f"All baseline EDA outputs saved to: {OUTPUT_DIR}")



8. Baseline preprocessing output
Saved model-ready baseline data: D:\Folder F\phamtuananh@23020010\UET.iSEML\2026.DAE.MLC Churn\Phase_0_EDA\outputs\preprocessed_baseline.csv
Preprocessed shape: 5000 rows x 74 columns
Recommended redundant features to review/drop later: total_day_charge, total_eve_charge, total_intl_charge, total_night_charge

Done
All baseline EDA outputs saved to: D:\Folder F\phamtuananh@23020010\UET.iSEML\2026.DAE.MLC Churn\Phase_0_EDA\outputs
